# Task 1 — Step 3: Dataset-Induced Length Confounding Study

### Motivation & Background
Preference datasets (like UltraFeedback) often suffer from **length bias**: human or AI annotators frequently prefer longer, more verbose responses, even when the extra text adds no substantive value.
This study tests:
1. Whether standard DPO preference gains are driven by length exploitation rather than content quality.
2. Whether training on a **length-balanced dataset** (equal parts `preferred_longer`, `matched_length`, `rejected_longer`) mitigates verbosity.
3. Explicit **word-limit constraint compliance** on `data/word_limit_prompts.jsonl`.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Length Analysis or Load Cached Diagnostics


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task1_dpo.analyze_length --config configs/dpo.yaml
else:
    print("[OK] Loading saved length confounding analysis...")


## 2. Stratified Preference Accuracy Breakdown


In [ ]:
len_file = "results/task1_dpo/dpo_length_analysis.json"
if os.path.exists(len_file):
    with open(len_file, encoding='utf-8') as f:
        len_data = json.load(f)
        
    std_strat = len_data.get("standard_dpo", {}).get("stratified", {})
    bal_strat = len_data.get("length_balanced_dpo", {}).get("stratified", {})
    
    strata = ["preferred_longer", "length_matched", "rejected_longer"]
    rows = []
    for s in strata:
        s_info = std_strat.get(s, {})
        b_info = bal_strat.get(s, {})
        rows.append({
            "Stratum": s.replace('_', ' ').title(),
            "Standard DPO Accuracy": f"{s_info.get('accuracy', 0)*100:.2f}%",
            "Length-Balanced DPO Accuracy": f"{b_info.get('accuracy', 0)*100:.2f}%",
            "Standard Margin": f"{s_info.get('mean_margin', 0):.4f}",
            "Balanced Margin": f"{b_info.get('mean_margin', 0):.4f}",
            "Pairs Count": s_info.get('total_pairs', 0)
        })
    df_strata = pd.DataFrame(rows)
    display(df_strata)


## 3. Explicit Word-Limit Compliance


In [ ]:
if os.path.exists(len_file):
    std_wl = len_data.get("standard_dpo", {}).get("word_limit", {})
    bal_wl = len_data.get("length_balanced_dpo", {}).get("word_limit", {})
    
    wl_df = pd.DataFrame([{
        "Condition": "Standard DPO",
        "Compliance Rate (%)": f"{std_wl.get('compliance_rate', 0)*100:.1f}%",
        "Mean Word Count": f"{std_wl.get('mean_word_count', 0):.1f}",
        "Target Word Count": "40 words"
    }, {
        "Condition": "Length-Balanced DPO",
        "Compliance Rate (%)": f"{bal_wl.get('compliance_rate', 0)*100:.1f}%",
        "Mean Word Count": f"{bal_wl.get('mean_word_count', 0):.1f}",
        "Target Word Count": "40 words"
    }])
    display(wl_df)


## 4. Visualizing Stratified Accuracy and Confounding Shifts


In [ ]:
if os.path.exists(len_file):
    labels = [r["Stratum"] for r in rows]
    std_accs = [float(r["Standard DPO Accuracy"].replace('%','')) for r in rows]
    bal_accs = [float(r["Length-Balanced DPO Accuracy"].replace('%','')) for r in rows]
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 1. Stratified Accuracy
    axes[0].bar(x - width/2, std_accs, width, label='Standard DPO', color='#3B82F6', alpha=0.85)
    axes[0].bar(x + width/2, bal_accs, width, label='Length-Balanced DPO', color='#10B981', alpha=0.85)
    axes[0].set_ylabel('Held-Out Accuracy (%)')
    axes[0].set_title('Preference Accuracy Across Length Strata')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # 2. Word-Limit Compliance
    wl_labels = ["Standard DPO", "Length-Balanced DPO"]
    wl_comps = [float(std_wl.get('compliance_rate', 0)*100), float(bal_wl.get('compliance_rate', 0)*100)]
    bars = axes[1].bar(wl_labels, wl_comps, color=['#3B82F6', '#10B981'], width=0.4, alpha=0.85)
    for b in bars:
        h = b.get_height()
        axes[1].text(b.get_x() + b.get_width()/2., h + 1, f"{h:.1f}%", ha='center', fontweight='bold')
    axes[1].set_ylabel('Compliance Rate (%)')
    axes[1].set_title('Explicit Word-Limit Compliance')
    axes[1].set_ylim(0, 100)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 5. Publication Dashboard Inspection


In [ ]:
p = "report/figures/task1_dpo_length_confounding.png"
if os.path.exists(p):
    print(f"Displaying publication figure: {p}")
    display(Image(filename=p, width=700))


## 6. Research Question 2 Analysis
- When preferred responses are shorter (`rejected_longer`), how did Standard DPO perform vs Length-Balanced DPO?
- Does length-balancing improve instruction following on concise requests?
